In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

C:\Users\Priyanshu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Priyanshu\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Priyanshu\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administ

In [5]:
seq_len = 8 
n_vocab = tokenizer.vocab_size
embed_dim = 2**6 
batch_size = 5

In [6]:
class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(n_vocab,embed_dim)
    self.positions = nn.Embedding(seq_len,embed_dim)
    self.finalLinear = nn.Linear(embed_dim,n_vocab,bias=False)       
    self.layernormA = nn.LayerNorm(embed_dim)             
    self.key   = nn.Linear(embed_dim,embed_dim,bias=False) 
    self.query = nn.Linear(embed_dim,embed_dim,bias=False) 
    self.value = nn.Linear(embed_dim,embed_dim,bias=False) 
    self.W0    = nn.Linear(embed_dim,embed_dim)            
    self.finalLinear.weight = nn.Parameter(self.embedding.weight)



  def forward(self,tokx):
    token_embed = self.embedding(tokx)
    posit_embed = self.positions(torch.arange(tokx.shape[-1]))
    x = token_embed + posit_embed
    x = self.layernormA(x)
    k = self.key(x)
    q = self.query(x)
    v = self.value(x)

    qk = q@k.transpose(-2,-1) 
    qk_scaled = qk * embed_dim**-.5 
    pastmask = torch.tril(torch.ones(x.shape[0],seq_len,seq_len)) 
    qk_scaled[pastmask==0] = -torch.inf
    qk_softmax = F.softmax(qk_scaled,dim=-1)
    y = qk_softmax @ v 

    y *= self.W0(y)
    y = self.finalLinear(y) / np.sqrt(embed_dim)

    return y, (pastmask,qk_scaled,qk_softmax)


  def generate(self,tokx,temperature=1,n_new_tokens=50):
    for _ in range(n_new_tokens):
      x = self(tokx[:,-seq_len:])[0] 
      x = x[:,-1,:] 
      probs = F.softmax(x/temperature,dim=-1) 
      tokx_next = torch.multinomial(probs,num_samples=1) 
      tokx = torch.cat( (tokx,tokx_next),dim=1) 
    return tokx


In [8]:

tokens = tokenizer.encode('I prefer oat milk in my coffee.')
X = torch.tensor(tokens[:-1]).unsqueeze(0)
y = torch.tensor(tokens[1:]).unsqueeze(0)

print(X.shape)
print(y.shape)
     

torch.Size([1, 8])
torch.Size([1, 8])


In [9]:
model = Model()
out,attn = model(X)

print(out.shape)

torch.Size([1, 8, 50257])


In [10]:

print(f'Expected loss for random weights: {-np.log(1/tokenizer.vocab_size):.3f}')
print(f'Observed mean log-softmax output: {torch.mean(-F.log_softmax(out.detach(),dim=-1)):.3f}')
print(f'Cross-entropy loss from pytorch:  {F.cross_entropy(out.view(-1, out.shape[-1]), y.view(-1)):.3f}')

Expected loss for random weights: 10.825
Observed mean log-softmax output: 10.828
Cross-entropy loss from pytorch:  10.846


In [11]:
print('Time-causal mask:\n',attn[0])
print('\nqk_scaled:\n',attn[1])
print('\nqk_softmax:\n',attn[2])

Time-causal mask:
 tensor([[[1., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1.]]])

qk_scaled:
 tensor([[[-4.0898e-01,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [-1.5370e-01,  3.2381e-02,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [ 1.4502e-01, -9.8118e-03,  2.9532e-02,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [ 4.6144e-01,  5.6347e-02,  6.9863e-02,  4.5573e-01,        -inf,
                 -inf,        -inf,        -inf],
         [ 4.3766e-01, -4.3853e-01,  3.5339e-01,  4.7457e-04,  1.6839e-01,
                 -inf,        -inf,        -inf

In [13]:
attn[2].detach().squeeze().sum(dim=1)


tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [14]:
text = 'When I grow up, I want to be a'
tokens = tokenizer.encode(text)
tokens = torch.tensor(tokens).unsqueeze(0)
generated_tokens = model.generate(tokens,temperature=2,n_new_tokens=10)[0]
tokenizer.decode(generated_tokens.tolist())

'When I grow up, I want to be aapons bubbles tradersescription760 Brill sides sprink permit haz'

In [ ]:
temps = [ .2, .7, 1, 2, 10 ]

for T in temps:
  tokz = model.generate(tokens,temperature=T,n_new_tokens=10)
  tokz = tokz[0].tolist()
  print(f'Temp = {T}:\n  {tokenizer.decode(tokz)}\n')

Temp = 0.2:
  When I grow up, I want to be a finals Santos riftwich flag receivers 429anneaterasu kn

Temp = 0.7:
  When I grow up, I want to be aGGGGGGGG ancestors times glossy highlighting Millascaristryoga enraged

Temp = 1:
  When I grow up, I want to be a stems thrilleracements foreskin'? Amb Northeast Arkansas Friday profitable

Temp = 2:
  When I grow up, I want to be a Weinstein hind nerves resonusb pilgrimage Find Menu€ Hits

Temp = 10:
  When I grow up, I want to be aonda struggle API bedroomsarted gigg miles docker honey elves

